<a href="https://colab.research.google.com/github/irum-zahra-awan/geneai/blob/main/PromptEngineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 2: Advanced Prompt Engineering

## Session Objectives:
- Explore prompt tuning methods: Chain-of-Thought (CoT), Tree-of-Thought (ToT)
- Understand system messages and few-shot learning

## Hands-On Activities:
1. Build ToT prompts using LangChain PromptTemplate
2. GPT-5 nano Prompt Refinement Exercises


<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/irum-zahra-awan/geneai/blob/main/PromptEngineering.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/irum-zahra-awan/geneai/blob/main/PromptEngineering.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>    

| Author |
| --- |
| [Irum Zahra](https://github.com/irum-zahra-awan/) |

## Setup and Installation

First, let's install the required packages and set up our environment.?

In [1]:
# Install required packages
# Run this cell first to install dependencies
!pip install -U langchain-core langchain-community langchain
!pip install openai langchain-openai python-dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 496.3/496.3 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.2/111.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.8
    Uninstalling langchain-core-1.2.8:
      Successfully uninstalled langchain-core-1.2.8
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.8
    Uninstalling langchain-1.2.8:
      Successfully uninstalled langchain-1.2.8
ERROR: pip's depen

## Configuration

### Storing Secrets in Colab

1.  **Open the Secrets Panel:** In your Colab notebook, look for the 'key' icon (🔑) on the left sidebar. Click it to open the 'Secrets' panel.
2.  **Add a New Secret:** Click the '+ New secret' button.
3.  **Name your Secret:** In the 'Name' field, enter a name for your secret (e.g., `MY_API_KEY`). This is how you'll refer to it in your code.
4.  **Enter the Secret Value:** In the 'Value' field, paste or type your sensitive information (e.g., your API key).
5.  **Save:** Click 'Done'.
6.  **Enable Notebook Access:** Make sure the 'Notebook access' toggle next to your secret is enabled for the current notebook.

### Accessing Secrets in Code

Once you've stored your secret, you can access it in your Python code using `google.colab.userdata`:

In [2]:
# Import the userdata module
from google.colab import userdata

# Access your secret by its name
subscription_key = userdata.get('OPENAI_API_KEY_AZURE')
endpoint = userdata.get('ENDPOINT_AZURE')

# You can now use 'subscription_key' AND 'endpoint' in your code without exposing its value directly.

### Setup Client

In [3]:
import os
from openai import AzureOpenAI

# from dotenv import load_dotenv
# load_dotenv()
# subscription_key=os.getenv("OPENAI_API_KEY")
# endpoint=os.getenv("endpoint")

model_name = "gpt-5-nano"
deployment = "gpt-5-nano"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# Part 1: Chain-of-Thought (CoT) Prompting with LangChain

Chain-of-Thought helps the model reason step-by-step, improving accuracy for complex problems.

##### 1. WITHOUT CHAIN-OF-THOUGHT

In [4]:
# Without CoT
def solve_without_cot():
    """
    Solve a problem without Chain-of-Thought
    The model might jump to conclusions
    """
    prompt = """
A store has 15 apples. They sell 60% in the morning and then get a
shipment that doubles their remaining apples. How many apples do they have now?
"""

    response = client.chat.completions.create(
        model=deployment, #"gpt-4o",
        messages=[{"role": "user", "content": prompt}]
    )
    usage = response.usage
    print(f"\n--- Token Usage ---")
    print(f"Input tokens (prompt): {usage.prompt_tokens}")
    print(f"Output tokens (completion): {usage.completion_tokens}")
    print(f"Total tokens: {usage.total_tokens}")
    print(f"-------------------\n")
    return response.choices[0].message.content

print("=== WITHOUT CHAIN-OF-THOUGHT ===")
print(solve_without_cot())
print("\n" + "="*60 + "\n")

=== WITHOUT CHAIN-OF-THOUGHT ===

--- Token Usage ---
Input tokens (prompt): 42
Output tokens (completion): 257
Total tokens: 299
-------------------

12

Explanation:
- Start with 15 apples.
- Sell 60%: 0.60 × 15 = 9 sold, leaving 15 − 9 = 6 apples.
- Shipment doubles remaining: 2 × 6 = 12 apples.




##### 2. WITH CHAIN-OF-THOUGHT

In [5]:
# With CoT
def solve_with_cot():
    """
    Solve the same problem with Chain-of-Thought prompting
    Explicitly ask for step-by-step reasoning
    """
    prompt = """
A store has 15 apples. They sell 60% in the morning and then get a
shipment that doubles their remaining apples. How many apples do they have now?

Let's solve this step-by-step:
1. First, calculate how many apples were sold
2. Then, find how many apples remain after the sale
3. Finally, calculate how many apples they have after the shipment doubles the remaining

Show your work for each step.
"""

    response = client.chat.completions.create(
        model=deployment, #"gpt-4o",
        messages=[{"role": "user", "content": prompt}]
    )
    usage = response.usage
    print(f"\n--- Token Usage ---")
    print(f"Input tokens (prompt): {usage.prompt_tokens}")
    print(f"Output tokens (completion): {usage.completion_tokens}")
    print(f"Total tokens: {usage.total_tokens}")
    print(f"-------------------\n")
    return response.choices[0].message.content

print("=== WITH CHAIN-OF-THOUGHT ===")
print(solve_with_cot())
print("\n" + "="*60 + "\n")


=== WITH CHAIN-OF-THOUGHT ===

--- Token Usage ---
Input tokens (prompt): 97
Output tokens (completion): 429
Total tokens: 526
-------------------

- Step 1: Calculate how many apples were sold
  60% of 15 = (60/100) * 15 = 9 apples sold.

- Step 2: Find how many apples remain after the sale
  15 total - 9 sold = 6 apples remaining.

- Step 3: Calculate how many apples after the shipment doubles the remaining
  6 remaining * 2 = 12 apples.

Answer: They have 12 apples now.




---

# Part 2: Tree-of-Thought (ToT) Prompting with LangChain

## What is Tree-of-Thought (ToT)?

Tree-of-Thought is an advanced prompting technique that:
- Generates multiple reasoning paths (branches)
- Evaluates each path
- Selects the best solution
- Useful for complex problem-solving that requires exploration of different approaches

### Comparison:
- **Chain-of-Thought (CoT)**: Linear reasoning (A → B → C → Solution)
- **Tree-of-Thought (ToT)**: Branching reasoning (explores multiple paths and selects best)

##### 1. Create a prompt template

In [6]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_openai import ChatOpenAI

# Step 1: Create a prompt template for generating multiple solution paths

# This template asks the LLM to think of different ways to solve the problem

tot_generation_template = PromptTemplate(
    input_variables=["numbers"],
    template="""
You are solving the Game of 24. You have these numbers: {numbers}
Goal: Use these 4 numbers and operations (+, -, *, /) to make 24.

Generate 3 DIFFERENT approaches to solve this problem.
For each approach, think step-by-step but don't solve completely yet.

Format your response as:
Approach 1: [describe the strategy]
Approach 2: [describe the strategy]
Approach 3: [describe the strategy]
"""
)

# Test the template
test_numbers = "4, 6, 8, 3"
prompt = tot_generation_template.format(numbers=test_numbers)

print("=== Generated Prompt ===")
print(prompt)
print("\n" + "="*50 + "\n")

=== Generated Prompt ===

You are solving the Game of 24. You have these numbers: 4, 6, 8, 3
Goal: Use these 4 numbers and operations (+, -, *, /) to make 24.

Generate 3 DIFFERENT approaches to solve this problem.
For each approach, think step-by-step but don't solve completely yet.

Format your response as:
Approach 1: [describe the strategy]
Approach 2: [describe the strategy]
Approach 3: [describe the strategy]





##### 2. Generate multiple approaches

In [7]:
# Step 2: Generate multiple approaches using GPT

def generate_tot_approaches(numbers):
    """
    Generate multiple approaches for solving the Game of 24

    Args:
        numbers: String of 4 numbers separated by commas

    Returns:
        String containing 3 different approaches
    """
    # Format the prompt with our numbers
    prompt = tot_generation_template.format(numbers=numbers)

    # Call GPT-4o to generate approaches
    response = client.chat.completions.create(
        model=deployment,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=1  # Higher temperature for more creative approaches
    )

    usage = response.usage
    print(f"\n--- Token Usage ---")
    print(f"Input tokens (prompt): {usage.prompt_tokens}")
    print(f"Output tokens (completion): {usage.completion_tokens}")
    print(f"Total tokens: {usage.total_tokens}")
    print(f"-------------------\n")

    return response.choices[0].message.content

# Generate approaches
approaches = generate_tot_approaches(test_numbers)
print("=== Generated Approaches ===")
print(approaches)
print("\n" + "="*50 + "\n")


--- Token Usage ---
Input tokens (prompt): 113
Output tokens (completion): 9118
Total tokens: 9231
-------------------

=== Generated Approaches ===
Approach 1: Exhaustive pairwise reduction (binary-tree search) with pruning
- Step 1: Pick any two of the numbers and apply +, -, *, or / (taking care with order for - and /). Record both the numeric result and the expression that produced it.
- Step 2: With that result and one of the remaining numbers, again apply all operations to form a new set of results. Record new values and their expressions.
- Step 3: Repeat once more with the last remaining number to form final results, checking if any path yields 24.
- Step 4: Use pruning to avoid duplicates: skip commutative duplicates (e.g., a+b vs b+a) and avoid reusing the same multiset of values in the same way.
- Step 5: If you find a path that gives 24, reconstruct the full expression from the recorded steps. If not, backtrack and try alternative pairings/orders.

Approach 2: Factorizatio

##### 3. Evaluate each approach

In [8]:
# Step 3: Evaluate each approach

# Create evaluation prompt template
tot_evaluation_template = PromptTemplate(
    input_variables=["numbers", "approaches"],
    template="""
You are evaluating different approaches to solve the Game of 24 with numbers: {numbers}

Here are the proposed approaches:
{approaches}

For each approach:
1. Try to execute it step-by-step
2. Rate its likelihood of success (High/Medium/Low)
3. Explain why

Format:
Approach 1 Evaluation: [rating] - [explanation]
Approach 2 Evaluation: [rating] - [explanation]
Approach 3 Evaluation: [rating] - [explanation]
"""
)

def evaluate_tot_approaches(numbers, approaches):
    """
    Evaluate the generated approaches

    Args:
        numbers: String of 4 numbers
        approaches: String containing the approaches to evaluate

    Returns:
        String containing evaluation of each approach
    """
    prompt = tot_evaluation_template.format(numbers=numbers, approaches=approaches)

    response = client.chat.completions.create(
        model=deployment, #"gpt-4o",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=1  #0.3 Lower temperature for consistent evaluation
    )

    return response, response.choices[0].message.content

# Evaluate the approaches
response, evaluation = evaluate_tot_approaches(test_numbers, approaches)
print("=== Approach Evaluation ===")
print(evaluation)
print("\n" + "="*50 + "\n")

usage = response.usage
print(f"\n--- Token Usage ---")
print(f"Input tokens (prompt): {usage.prompt_tokens}")
print(f"Output tokens (completion): {usage.completion_tokens}")
print(f"Total tokens: {usage.total_tokens}")
print(f"-------------------\n")

=== Approach Evaluation ===
Approach 1 Evaluation: High - Step-by-step demonstration and reasoning
- Start with numbers: {4, 6, 8, 3}
- Step 1 (pick pair 6 and 3): compute all results
  - 6+3 = 9, expression (6+3)
  - 6-3 = 3, expression (6-3)
  - 3-6 = -3, expression (3-6)
  - 6*3 = 18, expression (6*3)
  - 6/3 = 2, expression (6/3)
  - 3/6 = 0.5, expression (3/6)
- Step 2 (pick pair 8 and 4 from remaining): compute all results
  - 8+4 = 12, (8+4)
  - 8-4 = 4, (8-4)
  - 4-8 = -4, (4-8)
  - 8*4 = 32, (8*4)
  - 8/4 = 2, (8/4)
  - 4/8 = 0.5, (4/8)
- Step 3 (combine one result from Step 1 with one from Step 2)
  - For example, 2 (from 6/3) and 12 (from 8+4): 12 * 2 = 24
- Step 4 (pruning): skip commutative duplicates (e.g., we wouldn’t need to also try 12*2 vs 2*12 as distinct paths) and avoid reusing the same multiset in the same way
- Step 5 (reconstruction): final expression (6/3) * (8+4) = 24
- Likelihood of success: High
- Why: The problem has a straightforward 2-2 split path that im

##### 4. Select and execute the best approach

In [9]:
# Step 4: Select and execute the best approach

tot_solution_template = PromptTemplate(
    input_variables=["numbers", "approaches", "evaluation"],
    template="""
You are solving the Game of 24 with numbers: {numbers}

Proposed approaches:
{approaches}

Evaluation:
{evaluation}

Based on the evaluation, select the BEST approach and solve the problem step-by-step.
Show your complete calculation to reach 24.

Format:
Selected Approach: [which one]
Step-by-step Solution:
[show all calculations]
Final Answer: [the equation that equals 24]
"""
)

def solve_with_best_approach(numbers, approaches, evaluation):
    """
    Select the best approach and solve the problem

    Args:
        numbers: String of 4 numbers
        approaches: String containing approaches
        evaluation: String containing evaluations

    Returns:
        String containing the final solution
    """
    prompt = tot_solution_template.format(
        numbers=numbers,
        approaches=approaches,
        evaluation=evaluation
    )

    response = client.chat.completions.create(
        model=deployment, #"gpt-4o",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=1 #0.2  # Low temperature for accurate calculation
    )

    return response, response.choices[0].message.content

# Get the final solution
response, solution = solve_with_best_approach(test_numbers, approaches, evaluation)
print("=== Final Solution ===")
print(solution)
print("\n" + "="*50 + "\n")

usage = response.usage
print(f"\n--- Token Usage ---")
print(f"Input tokens (prompt): {usage.prompt_tokens}")
print(f"Output tokens (completion): {usage.completion_tokens}")
print(f"Total tokens: {usage.total_tokens}")
print(f"-------------------\n")


=== Final Solution ===
Selected Approach: Approach 2

Step-by-step Solution:
- Step 1: Partition the four numbers into two pairs: {6, 3} and {8, 4}.
- Step 2: Compute all possible values for each pair.
  - For {6, 3}: 6+3 = 9; 6-3 = 3; 3-6 = -3; 6*3 = 18; 6/3 = 2; 3/6 = 0.5
  - For {8, 4}: 8+4 = 12; 8-4 = 4; 4-8 = -4; 8*4 = 32; 8/4 = 2; 4/8 = 0.5
- Step 3: Combine a value from the first pair with a value from the second pair using a simple operation to reach 24.
  - Take L = 2 (from 6/3) and R = 12 (from 8+4); 2 * 12 = 24
- Step 4: Reconstruct the full expression that yields 24.
  - (6/3) * (8+4) = 24

Final Answer: (6/3) * (8+4) = 24



--- Token Usage ---
Input tokens (prompt): 1797
Output tokens (completion): 1187
Total tokens: 2984
-------------------



##### Complete ToT Pipeline Function

In [10]:
# Step 5: Complete ToT Pipeline Function

def tree_of_thought_solver(numbers):
    """
    Complete Tree-of-Thought pipeline for Game of 24

    This function:
    1. Generates multiple approaches
    2. Evaluates each approach
    3. Selects and executes the best one

    Args:
        numbers: String of 4 numbers (e.g., "4, 6, 8, 3")

    Returns:
        Dictionary with approaches, evaluation, and solution
    """
    print(f"Solving Game of 24 for numbers: {numbers}")
    print("\nStep 1: Generating approaches...")

    # Generate approaches
    approaches = generate_tot_approaches(numbers)
    print("✓ Approaches generated\n")

    print("Step 2: Evaluating approaches...")
    # Evaluate approaches
    evaluation = evaluate_tot_approaches(numbers, approaches)
    print("✓ Evaluation complete\n")

    print("Step 3: Solving with best approach...")
    # Get solution
    solution = solve_with_best_approach(numbers, approaches, evaluation)
    print("✓ Solution found\n")

    return {
        "approaches": approaches,
        "evaluation": evaluation,
        "solution": solution
    }

# Test with different numbers
print("="*60)
print("TREE-OF-THOUGHT DEMONSTRATION")
print("="*60)

result = tree_of_thought_solver("3, 7, 8, 8")

print("\n" + "="*60)
print("FINAL SOLUTION:")
print("="*60)
print(result["solution"])

TREE-OF-THOUGHT DEMONSTRATION
Solving Game of 24 for numbers: 3, 7, 8, 8

Step 1: Generating approaches...

--- Token Usage ---
Input tokens (prompt): 113
Output tokens (completion): 13878
Total tokens: 13991
-------------------

✓ Approaches generated

Step 2: Evaluating approaches...
✓ Evaluation complete

Step 3: Solving with best approach...
✓ Solution found


FINAL SOLUTION:
(ChatCompletion(id='chatcmpl-D78Q05xnmGIZirEDrEViaFGH0FOSc', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Selected Approach: Approach 3\n\nStep-by-step Solution:\n- Use backward reasoning: final operation is multiplication, combining a subexpression from {7, 8, 8} and the remaining number 3.\n- Compute 8 ÷ 8 = 1\n- Then 7 + 1 = 8\n- Finally 8 × 3 = 24\n\nFinal Answer: (7 + 8/8) × 3 = 24', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), content_filter_results={'hate': {'filtered': False, 'severity': 'safe'}

---

# Part 3: Prompt Refinement Exercises

## What is Prompt Refinement?

Prompt refinement is the iterative process of improving prompts to get better outputs. We'll practice:
1. Starting with a basic prompt
2. Identifying issues
3. Refining the prompt
4. Comparing results

## Exercise 3.1: From Basic to Advanced - Email Writing

##### 1. Basic Prompt

In [11]:
# Version 1: Basic Prompt (Vague)

def test_prompt_v1():
    """
    Test a basic, vague prompt
    Problem: Too generic, no context
    """
    prompt = "Write an email to a client."

    response = client.chat.completions.create(
        model=deployment, #"gpt-4o",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    usage = response.usage
    print(f"\n--- Token Usage ---")
    print(f"Input tokens (prompt): {usage.prompt_tokens}")
    print(f"Output tokens (completion): {usage.completion_tokens}")
    print(f"Total tokens: {usage.total_tokens}")
    print(f"-------------------\n")
    return response.choices[0].message.content

print("=== VERSION 1: Basic Prompt ===")
print("Prompt: 'Write an email to a client.'")
print("\nOutput:")
result_v1 = test_prompt_v1()
print(result_v1)
print("\n" + "="*60 + "\n")

=== VERSION 1: Basic Prompt ===
Prompt: 'Write an email to a client.'

Output:

--- Token Usage ---
Input tokens (prompt): 13
Output tokens (completion): 1298
Total tokens: 1311
-------------------

Here’s a ready-to-use professional email you can customize. It’s suitable for a general client update or follow-up.

Subject: Update on [Project/Order] and Next Steps

Dear [Client Name],

I hope you’re doing well. I’m writing to provide a brief update on [Project/Order] and outline the next steps.

- Current status: [Brief status update].
- What’s been completed: [Key milestones or deliverables].
- Next steps and timeline: [Upcoming tasks and estimated dates].
- Any requests from you: [If you need authorization, input, or documents].

If you’d like to discuss any of these points, I’m happy to arrange a brief call. I’m available at [Your Availability] or feel free to suggest a time that works for you.

Thank you for your continued partnership. Please don’t hesitate to reach out with any que

##### 2. Adding Some Context

In [12]:
# Version 2: Adding Context

def test_prompt_v2():
    """
    Improved prompt with context
    Improvement: Added purpose and context
    """
    prompt = """
Write an email to a client informing them that their project delivery
will be delayed by 2 weeks due to technical challenges.
"""

    response = client.chat.completions.create(
        model=deployment, #"gpt-4o",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    usage = response.usage
    print(f"\n--- Token Usage ---")
    print(f"Input tokens (prompt): {usage.prompt_tokens}")
    print(f"Output tokens (completion): {usage.completion_tokens}")
    print(f"Total tokens: {usage.total_tokens}")
    print(f"-------------------\n")
    return response.choices[0].message.content

print("=== VERSION 2: Added Context ===")
print("Improvement: Added purpose and context")
print("\nOutput:")
result_v2 = test_prompt_v2()
print(result_v2)
print("\n" + "="*60 + "\n")

=== VERSION 2: Added Context ===
Improvement: Added purpose and context

Output:

--- Token Usage ---
Input tokens (prompt): 32
Output tokens (completion): 1813
Total tokens: 1845
-------------------

Subject: Update on [Project Name] Delivery Timeline

Dear [Client Name],

I hope you’re well. I’m writing to inform you that, due to technical challenges encountered in [brief area or system], we will need to extend the delivery of [Project Name] by two weeks. We identified the issue on [date], and while we’ve made substantial progress, additional time is required to ensure the solution meets our quality standards and delivers the expected reliability.

New delivery date: [Month Day, Year] (14 days later than the original date of [Original Date]).

What this means for you:
- The project will be delivered with the same scope and quality, but with a revised timeline.
- We will provide weekly status updates and a current risk log to keep you informed of progress and any new developments.

Wh

##### 3. Adding Tone and Structure

In [13]:
# Version 3: Adding Tone and Structure

def test_prompt_v3():
    """
    Further improved prompt with tone and structure
    Improvement: Specified tone, format, and key points
    """
    prompt = """
Write a professional and empathetic email to a client informing them
that their project delivery will be delayed by 2 weeks due to technical challenges.

Tone: Professional but empathetic
Include:
- Brief explanation of the delay
- Reassurance about quality
- New timeline
- Offer of a brief call to discuss

Keep it concise (under 150 words).
"""

    response = client.chat.completions.create(
        model=deployment, #"gpt-4o",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    usage = response.usage
    print(f"\n--- Token Usage ---")
    print(f"Input tokens (prompt): {usage.prompt_tokens}")
    print(f"Output tokens (completion): {usage.completion_tokens}")
    print(f"Total tokens: {usage.total_tokens}")
    print(f"-------------------\n")

    return response.choices[0].message.content

print("=== VERSION 3: Added Tone and Structure ===")
print("Improvement: Specified tone, format, and key points")
print("\nOutput:")
result_v3 = test_prompt_v3()
print(result_v3)
print("\n" + "="*60 + "\n")

=== VERSION 3: Added Tone and Structure ===
Improvement: Specified tone, format, and key points

Output:

--- Token Usage ---
Input tokens (prompt): 81
Output tokens (completion): 2326
Total tokens: 2407
-------------------

Subject: Update on project delivery timeline

Dear [Client Name],

I’m writing to inform you that we’ve encountered technical challenges during the integration phase that require additional time to resolve. I understand this may be inconvenient, and I appreciate your patience as we work to deliver a robust solution. This delay extends the timeline by two weeks.

New delivery date: February 22, 2026.

We’re committed to quality and will use the extra time for thorough testing to ensure a reliable product. If you’d like, we can schedule a brief 15-minute call to discuss the details at your convenience.

Thank you for your understanding.

Best regards,
[Your Name]
[Your Title]
[Company]




##### 4. Few-Shot Examples

In [14]:
# Version 4: Using LangChain PromptTemplate with Few-Shot Examples

def test_prompt_v4():
    """
    Best practice prompt using template and few-shot learning
    Improvement: Added examples and structured template
    """

    # Create a comprehensive prompt template
    email_template = PromptTemplate(
        input_variables=["situation", "delay_period", "reason"],
        template="""
You are a professional project manager writing to a valued client.

Situation: {situation}
Delay Period: {delay_period}
Reason: {reason}

Example of good client communication:
"Dear [Client], I wanted to reach out regarding [project]. Due to [brief reason],
we need to adjust our timeline by [period]. We're committed to delivering excellent
results and this additional time ensures we meet our quality standards. The new
delivery date is [date]. I'm available to discuss this at your convenience."

Write a professional email following this example's tone and structure.
Keep it under 150 words, empathetic, and solution-focused.
"""
    )

    # Fill in the template
    prompt = email_template.format(
        situation="project delivery delay notification",
        delay_period="2 weeks",
        reason="unexpected technical integration challenges"
    )

    response = client.chat.completions.create(
        model=deployment,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    usage = response.usage
    print(f"\n--- Token Usage ---")
    print(f"Input tokens (prompt): {usage.prompt_tokens}")
    print(f"Output tokens (completion): {usage.completion_tokens}")
    print(f"Total tokens: {usage.total_tokens}")
    print(f"-------------------\n")
    return response.choices[0].message.content

print("=== VERSION 4: Template + Few-Shot Examples ===")
print("Improvement: Added examples and structured template")
print("\nOutput:")
result_v4 = test_prompt_v4()
print(result_v4)
print("\n" + "="*60 + "\n")

=== VERSION 4: Template + Few-Shot Examples ===
Improvement: Added examples and structured template

Output:

--- Token Usage ---
Input tokens (prompt): 138
Output tokens (completion): 1640
Total tokens: 1778
-------------------

Dear [Client], I wanted to reach out regarding [Project Name]. Due to unexpected technical integration challenges, we need to adjust our timeline by two weeks. We're committed to delivering excellent results and this additional time ensures we meet our quality standards. The new delivery date is February 22, 2026. To minimize impact, we’ve realigned priorities, increased hands-on engineering effort, and will provide weekly progress updates. I’m available to discuss this at your convenience.




---

# Part 4: System Role Instructions - Building Custom Assistants

## What are System Messages?

System messages:
- Define the assistant's behavior and personality
- Set guidelines for responses
- Persist across the entire conversation
- Are different from user messages

## Message Roles:
- **system**: Instructions for the assistant's behavior (set once)
- **user**: Messages from the user
- **assistant**: Responses from the AI

## Lab 4.1: Customer Support Assistant

In [15]:
# Create a customer support assistant with specific behavior

def create_support_assistant(user_message):
    """
    Customer support assistant with defined personality and rules

    Args:
        user_message: The customer's message

    Returns:
        Assistant's response
    """

    # Define the system message - this sets the assistant's behavior
    system_message = """
You are a friendly and professional customer support assistant for TechStore,
an online electronics retailer.

Your responsibilities:
- Help customers with order inquiries, returns, and technical questions
- Be empathetic and patient
- If you cannot solve an issue, offer to escalate to a human agent
- Always ask for order number when relevant
- Keep responses concise but helpful (under 100 words unless more detail is needed)

Tone: Friendly, professional, solution-oriented
Never: Make promises you can't keep, share customer data, or get defensive
"""

    # Make the API call with system and user messages
    response = client.chat.completions.create(
        model=deployment,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
        ]
        #,temperature=0.7
    )
    usage = response.usage
    print(f"\n--- Token Usage ---")
    print(f"Input tokens (prompt): {usage.prompt_tokens}")
    print(f"Output tokens (completion): {usage.completion_tokens}")
    print(f"Total tokens: {usage.total_tokens}")
    print(f"-------------------\n")

    return response.choices[0].message.content

# Test the assistant with different queries
test_queries = [
    "My package hasn't arrived yet and it's been 2 weeks!",
    "How do I return a defective laptop?",
    "Can you tell me about your warranty policy?"
]

print("=== CUSTOMER SUPPORT ASSISTANT DEMO ===")
print("\n")

for i, query in enumerate(test_queries, 1):
    print(f"Customer Query {i}: {query}")
    print(f"\nAssistant Response:")
    response = create_support_assistant(query)
    print(response)
    print("\n" + "="*60 + "\n")

=== CUSTOMER SUPPORT ASSISTANT DEMO ===


Customer Query 1: My package hasn't arrived yet and it's been 2 weeks!

Assistant Response:

--- Token Usage ---
Input tokens (prompt): 131
Output tokens (completion): 981
Total tokens: 1112
-------------------

I’m sorry your package hasn’t arrived after two weeks—that’s frustrating. Please share your order number (and tracking number if you have it). I’ll check the latest status with the carrier and start an investigation. Depending on the outcome, we can discuss options like a replacement or refund. If you’d like, I can escalate this to a human agent for faster resolution.


Customer Query 2: How do I return a defective laptop?

Assistant Response:

--- Token Usage ---
Input tokens (prompt): 127
Output tokens (completion): 1543
Total tokens: 1670
-------------------

Sorry to hear your laptop is defective—we’ll get this sorted. Please share your TechStore order number.

Steps to start the return:
- Gather laptop, charger, original box, and a

## Lab 4.2: Multi-Turn Conversation with Memory

In [16]:
# Build a conversational assistant that remembers context

def chat_with_memory(conversation_history, new_user_message, system_prompt):
    """
    Chat function that maintains conversation history

    Args:
        conversation_history: List of previous messages
        new_user_message: The new message from user
        system_prompt: System instructions for the assistant

    Returns:
        Tuple of (assistant_response, updated_history)
    """

    # Add the new user message to history
    conversation_history.append({
        "role": "user",
        "content": new_user_message
    })

    # Create messages list: system message + conversation history
    messages = [
        {"role": "system", "content": system_prompt}
    ] + conversation_history

    # Get response from GPT-4o
    response = client.chat.completions.create(
        model=deployment,
        messages=messages,
        #temperature=0.7
    )
    usage = response.usage
    print(f"\n--- Token Usage ---")
    print(f"Input tokens (prompt): {usage.prompt_tokens}")
    print(f"Output tokens (completion): {usage.completion_tokens}")
    print(f"Total tokens: {usage.total_tokens}")
    print(f"-------------------\n")
    assistant_message = response.choices[0].message.content

    # Add assistant's response to history
    conversation_history.append({
        "role": "assistant",
        "content": assistant_message
    })

    return assistant_message, conversation_history

# Example: Personal Tutor Assistant
tutor_system_prompt = """
You are a patient and encouraging Python programming tutor.

Your approach:
- Ask questions to understand the student's level
- Explain concepts with simple examples
- Encourage practice and experimentation
- Remember what the student has learned in this conversation
- Adjust difficulty based on student's responses

Teaching style: Socratic method - ask guiding questions rather than giving direct answers
"""

# Initialize conversation
history = []

print("=== PYTHON TUTOR CONVERSATION DEMO ===")
print("(Notice how the assistant remembers context across messages)\n")

# Message 1
print("Student: I want to learn about Python lists")
response1, history = chat_with_memory(
    history,
    "I want to learn about Python lists",
    tutor_system_prompt
)
print(f"Tutor: {response1}\n")
print("="*60 + "\n")

# Message 2 - references previous context
print("Student: I'm a complete beginner")
response2, history = chat_with_memory(
    history,
    "I'm a complete beginner",
    tutor_system_prompt
)
print(f"Tutor: {response2}\n")
print("="*60 + "\n")

# Message 3 - builds on conversation
print("Student: Can you give me an example?")
response3, history = chat_with_memory(
    history,
    "Can you give me an example?",
    tutor_system_prompt
)
print(f"Tutor: {response3}\n")
print("="*60 + "\n")

print(f"\nConversation length: {len(history)} messages")

=== PYTHON TUTOR CONVERSATION DEMO ===
(Notice how the assistant remembers context across messages)

Student: I want to learn about Python lists

--- Token Usage ---
Input tokens (prompt): 89
Output tokens (completion): 1180
Total tokens: 1269
-------------------

Tutor: Great! Lists are a core feature in Python. Before we dive in, can I ask a couple quick questions to match the pace?

- Have you written Python before, even a little?
- Do you have Python installed and a place to run small snippets (like a REPL, Jupyter, or a simple script)?
- Do you want to focus on basics first, or jump into some hands-on practice rights away?

If you’re totally new to Python, here’s a gentle plan we can follow:
- What a list is: an ordered, mutable collection of items.
- Create a list and access items by their position (index).
- Change items, and add or remove items.
- Common handy operations: length, looping through, and a few common methods (append, extend, insert, remove, pop).
- A quick intro to

## Lab 4.3: Building Different Assistant Personalities

In [17]:
# Compare different system prompts for the same query

# user question
user_question = "Explain what machine learning is"

# Different system prompts create different personalities

# Assistant 1: Formal Academic
academic_system = """
You are a university professor specializing in computer science.
Use precise, academic language with technical terminology.
Structure explanations formally with clear definitions.
"""

# Assistant 2: Casual Explainer
casual_system = """
You are a friendly tech enthusiast explaining concepts to a friend.
Use simple language, everyday analogies, and a conversational tone.
Make complex topics feel approachable and fun.
"""

# Assistant 3: Socratic Teacher
socratic_system = """
You are a teacher who uses the Socratic method.
Instead of explaining directly, ask thought-provoking questions.
Guide the student to discover the answer themselves.
"""

def get_response_with_personality(system_prompt, user_msg):
    """Get response with specific personality"""
    response = client.chat.completions.create(
        model= deployment,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_msg}
        ],
        #temperature=0.7
    )
    usage = response.usage
    print(f"\n--- Token Usage ---")
    print(f"Input tokens (prompt): {usage.prompt_tokens}")
    print(f"Output tokens (completion): {usage.completion_tokens}")
    print(f"Total tokens: {usage.total_tokens}")
    print(f"-------------------\n")
    return response.choices[0].message.content

print("=== SAME QUESTION, DIFFERENT PERSONALITIES ===")
print(f"\nUser Question: {user_question}\n")
print("="*60)

# Get responses from each personality
print("\n1. FORMAL ACADEMIC RESPONSE:")
print("="*60)
print(get_response_with_personality(academic_system, user_question))

print("\n\n2. CASUAL EXPLAINER RESPONSE:")
print("="*60)
print(get_response_with_personality(casual_system, user_question))

print("\n\n3. SOCRATIC TEACHER RESPONSE:")
print("="*60)
print(get_response_with_personality(socratic_system, user_question))

=== SAME QUESTION, DIFFERENT PERSONALITIES ===

User Question: Explain what machine learning is


1. FORMAL ACADEMIC RESPONSE:

--- Token Usage ---
Input tokens (prompt): 42
Output tokens (completion): 2250
Total tokens: 2292
-------------------

Machine learning is a discipline within computer science and statistics that studies how to construct algorithms capable of improving their performance on a task through experience with data, rather than being explicitly programmed for that task.

Formal framework (high-level definitions)
- Data: A collection D of examples, typically D = {(x1, y1), (x2, y2), ..., (xn, yn)}, where each xi is an input (feature vector) and yi is the corresponding target.
- Task: A function or predictor that maps inputs to outputs. This is modeled as a hypothesis h from a hypothesis class H, where h: X -> Y.
- Objective: Choose h to perform well on unseen data. This is typically formalized via a loss function L(y, h(x)) that quantifies how far the prediction h(x) 

## Lab 4.4: Advanced System Prompt - JSON Output Format

System prompts can also enforce output formatting, such as JSON.

In [18]:
# System prompt that enforces JSON output

import json

json_system_prompt = """
You are a data extraction assistant.
Extract information from text and return it in valid JSON format.

For restaurant reviews, extract:
- restaurant_name (string)
- rating (number 1-5)
- cuisine_type (string)
- price_range (string: "$", "$$", "$$$", or "$$$$")
- key_highlights (list of strings)
- would_recommend (boolean)

Always return ONLY valid JSON, no additional text.
"""

# Test with a restaurant review
review_text = """
I visited The Golden Spoon last night and it was amazing! This Italian restaurant
has the best homemade pasta I've ever had. The prices are a bit high - we spent
about $80 per person - but the quality is worth it. The ambiance is romantic and
the service was excellent. I'd definitely go back. I'd give it 5 stars!
"""

response = client.chat.completions.create(
    model=deployment,
    messages=[
        {"role": "system", "content": json_system_prompt},
        {"role": "user", "content": review_text}
    ]
    #,temperature=0.3  # Low temperature for consistent formatting
)

# Get the response
json_output = response.choices[0].message.content

usage = response.usage
print(f"\n--- Token Usage ---")
print(f"Input tokens (prompt): {usage.prompt_tokens}")
print(f"Output tokens (completion): {usage.completion_tokens}")
print(f"Total tokens: {usage.total_tokens}")
print(f"-------------------\n")

print("=== STRUCTURED JSON OUTPUT ===")
print("\nRaw Output:")
print(json_output)

# Parse and pretty-print the JSON
print("\n" + "="*60)
print("Parsed JSON:")
try:
    parsed_data = json.loads(json_output)
    print(json.dumps(parsed_data, indent=2))

    # You can now use this data programmatically
    print("\n" + "="*60)
    print("Accessing data:")
    print(f"Restaurant: {parsed_data['restaurant_name']}")
    print(f"Rating: {parsed_data['rating']}/5")
    print(f"Would recommend: {parsed_data['would_recommend']}")

except json.JSONDecodeError:
    print("Error: Output is not valid JSON")


--- Token Usage ---
Input tokens (prompt): 172
Output tokens (completion): 655
Total tokens: 827
-------------------

=== STRUCTURED JSON OUTPUT ===

Raw Output:
{
  "restaurant_name": "The Golden Spoon",
  "rating": 5,
  "cuisine_type": "Italian",
  "price_range": "$$$",
  "key_highlights": ["best homemade pasta", "romantic ambiance", "excellent service", "worth the price"],
  "would_recommend": true
}

Parsed JSON:
{
  "restaurant_name": "The Golden Spoon",
  "rating": 5,
  "cuisine_type": "Italian",
  "price_range": "$$$",
  "key_highlights": [
    "best homemade pasta",
    "romantic ambiance",
    "excellent service",
    "worth the price"
  ],
  "would_recommend": true
}

Accessing data:
Restaurant: The Golden Spoon
Rating: 5/5
Would recommend: True
